In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, embed_dim, max_len=100):
        super().__init__()
        
        # 1. Create a matrix of zeros with shape [max_len, embed_dim]
        pe = torch.zeros(max_len, embed_dim)

        # 2. Generate position indices [0, 1, 2, ..., max_len-1] as column vector
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 3. Calculate the internal exponential scaling term for denominators
        # We step by 2 because we assign sin and cos to even/odd pairs
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))

        # 4. Fill even dimensions with sine waves, odd dimensions with cosine waves
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 5. Add a batch dimension: [1, max_len, embed_dim]
        # register_buffer ensures this tensor is saved with the model but not trained as a weight
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):

        # x shape: [batch_size, seq_len, embed_dim]
        # We slice our precomputed PE matrix to match the current input sequence length
        x = x + self.pe[:, :x.size(1)]
        return x

In [3]:
# Instantiate and check shape
pe_layer = PositionalEncoding(embed_dim=4, max_len=10)
toy_input = torch.zeros(1, 3, 4) # Batch=1, Tokens=3, Features=4
toy_output = pe_layer(toy_input)

print("Input shape: ", toy_input.shape)
print("Output shape:", toy_output.shape)
print("\nValues added to our zero input (The PE vectors themselves):\n", toy_output)

Input shape:  torch.Size([1, 3, 4])
Output shape: torch.Size([1, 3, 4])

Values added to our zero input (The PE vectors themselves):
 tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])


# Exercise

In [5]:
import torch
import math

# A static 2-position, 4-dimension PE matrix
# Row 0 = Position 0, Row 1 = Position 1
PE = torch.tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],   # Waves at pos 0
                    [ 0.8415,  0.5403,  0.0100,  0.9999]])  # Waves at pos 1

# Imagine the word "AI" appears twice in our sentence at index 0 and index 1.
# Since it's the same word, it starts with the exact same embedding vector.
token_pos0 = torch.tensor([1.5, -0.5, 2.0, 0.0])
token_pos1 = torch.tensor([1.5, -0.5, 2.0, 0.0])

# 1. Compute the raw dot product between the original un-encoded tokens
# Hint: Use torch.dot()
raw_dot = torch.dot(token_pos0, token_pos1)
print("Raw Dot Product (Identical vectors):", raw_dot.item())

# 2. Extract position 0 vector and position 1 vector from the PE matrix
pe_vector0 = PE[0]
pe_vector1 = PE[1]

# 3. Add the positional signatures directly to the respective tokens
encoded_pos0 = token_pos0 + pe_vector0
encoded_pos1 = token_pos1 + pe_vector1

# 4. Compute the dot product between the two modified, position-aware tokens
encoded_dot = torch.dot(encoded_pos0, encoded_pos1)
print("Position-Aware Dot Product:        ", encoded_dot.item())

Raw Dot Product (Identical vectors): 6.5
Position-Aware Dot Product:         8.552299499511719
